# Can baseline physical activity improve prediction of later chronic pain?

## A logistic regression case study using NLSY97 data

**Governing Question**

Among eligible NLSY97 survey participants, does adding self-reported physical-activity information at the 2015-round interview alongside age and general health improve logistic regression predictions of chronic pain reported at the 2019-round interview?

**Road Map** (---TO COMPLETE---)

## 1. Investigation Contract  (---TO COMPLETE---)

- **Observation unit:** One eligible NLS97 survey participant

- **Population of interest:** NLSY97 participants with a valid 2015 baseline interview and a valid 2019 chronic pain response

- **Variables used:**

    <br>

    | Variables | CSV Column | Meaning |
    |---|---|---|
    | Identifier | `PUBID__1997` | Participant ID |
    | Baseline feature | `CV_AGE_INT_DATE__2015` | Age at baseline interview, in years |
    | Baseline feature | `YHEA-100__2015` | General health: `1` = excellent through `5` = poor |
    | Baseline feature | `YHEA-600AA__2015` | Vigorous-activity status |
    | Baseline feature | `YHEA-600BA__2015` | Light/moderate-activity status |
    | Baseline feature | `YHEA-600CA__2015` | Muscle-strengthening-activity status |
    | Target | `YHEA-PAIN-1__2019` | `1` = yes, `0` = no |

    <br>
    <br>

- **Type of case study:** Predictive and exploratory


## 2. Data Provenance and Fitness for Purpose

### 2.1 Dataset source (---TO COMPLETE---)


### 2.2 Variable definitions

The following variables are used to predict self-reported chronic pain in 2019 using information collected during the baseline 2015 NLSY97 interview.

#### Feature Variables (X)

| Feature Name | CSV Column Name | Variable Type | Units and Meanings |
|---|---|---|---|
| **Age** | `CV_AGE_INT_DATE__2015` | Numerical (discrete) | Age (in years) at the baseline interview |
| **General Health** | `YHEA-100__2015` | Ordinal categorical | `1` = Excellent; `2` = Very good; `3` = Good; `4` = Fair; `5` = Poor |
| **Vigorous Physical Activity** | `YHEA-600AA__2015` | Categorical (nominal) | `0` = Never; `1` = Participates; `2` = Unable to participate |
| **Light/Moderate Physical Activity** | `YHEA-600BA__2015` | Categorical (nominal) | `0` = Never; `1` = Participates; `2` = Unable to participate |
| **Muscle-Strengthening Activity** | `YHEA-600CA__2015` | Categorical (nominal) | `0` = Never; `1` = Participates; `2` = Unable to participate |

#### Target Variable (y)

| Target Name | CSV Column Name | Variable Type | Meaning |
|---|---|---|---|
| **Chronic Pain (2019)** | `YHEA-PAIN-1__2019` | Binary categorical | `0` = No chronic pain reported; `1` = Chronic pain reported |

The target records whether the respondent reported chronic pain caused by an illness or medical condition during the preceding 30 days. Note that it does not establish when that pain first developed.

#### Special Response Codes

The NLSY97 dataset uses negative values to identify missing, refused, skipped, or noninterview responses. In this case study, these codes are *not valid measurements* of age, health, physical activity, or chronic pain and will be removed during data cleaning. 

| Code | Meaning | Interpretation |
|---|---|---|
| `-1` | Refusal | Respondent refused to answer the question |
| `-2` | Don't know | Respondent did not know the answer |
| `-3` | Invalid skip | Question was not answered when a response was expected |
| `-4` | Valid skip | Question was legitimately skipped or was not applicable |
| `-5` | Noninterview | Respondent did not complete that survey interview |

#### Preprocessing Considerations

- **Age:** Treat as a numerical value after excluding invalid values

- **General health:** Treat as an integer categorical variable, where higher values indicate worse self-reported health

- **Physical activity:** Preserve the distinction between never participating (`0`) and being unable to participate (`2`)

- **Chronic pain:** Retain only valid binary target responses (`0` and `1`) 

- **Missing data:** Convert special negative codes to missing values where appropriate. Handle missing predictors during preprocessing

- **Feature encoding:** Encode categorical activity variables appropriately

### 2.3 Loading the dataset

In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from dsm_project.algorithms import (
    create_student_algorithm,
    registered_student_algorithms,
)
from dsm_project.config import load_config
from dsm_project.contracts import TaskKind
from dsm_project.data import load_project_dataset
from dsm_project.experiments import (
    evaluate_supervised_estimator,
    fit_supervised_candidate,
)

CONFIG_PATH = Path("../configs/nlsy97.toml")
TASK_KEY = "classification"
STUDENT_ALGORITHM = "batch_logistic_regression"
TRUSTED_REFERENCE = "logistic_regression"

config = load_config(CONFIG_PATH)
dataset = load_project_dataset(config)
task = dataset.tasks[TASK_KEY]
frame = dataset.observations

print("provenance:", dataset.provenance)
print("task:", task)
print("registered student algorithms:", registered_student_algorithms())

provenance: DatasetProvenance(name='Subset of the National Longitudinal Survey of Youth 1997 (NLSY97)', source='The National Longitudinal Survey of Youth 1997 (NLSY97) public-use dataset, downloaded 2026-11-01 from https://nlsinfo.org/content/access-data-investigator/accessing-data-cohorts', license='The NLSY97 dataset is public-use data. Follow official NLS guidance for access, citation, and redistribution.', observation_unit='One original NLSY97 participant (including nonrespondents)')
task: TaskDefinition(name='Predict Chronic Pain in 2019', kind=<TaskKind.CLASSIFICATION: 'classification'>, feature_columns=('CV_AGE_INT_DATE__2015', 'YHEA-100__2015', 'YHEA-600AA__2015', 'YHEA-600BA__2015', 'YHEA-600CA__2015'), target_column='YHEA-PAIN-1__2019', metric='balanced_accuracy')
registered student algorithms: ('batch_linear_regression',)


## 3. Purposeful Exploration

### 3.1 Prepare the eligible respondants

In [3]:
# Identify variables
id_col = "PUBID__1997"
feature_cols = list(task.feature_columns)
target_col = task.target_column

# Identify respondents who completed 2015 and 2019 rounds
baseline_2015 = frame["CV_INTERVIEW_DATE~Y__2015"].isin([2015, 2016])
followup_2019 = frame["CV_INTERVIEW_DATE~Y__2019"].isin([2019, 2020])

# Require valid binary chronic-pain outcome (remove negative codes)
valid_target = frame[target_col].isin([0, 1])

# Make sure baseline interview happened first
valid_time = (
    frame["CV_INTERVIEW_CMONTH__2015"]
    < frame["CV_INTERVIEW_CMONTH__2019"]
)

# Count how many respondents remain at each step
cohort_counts = pd.DataFrame({
    "Stage": [
        "Original respondents",
        "Completed 2015 interview",
        "Completed both interviews",
        "Valid 2019 chronic-pain response",
        "Valid interview chronology",
    ],
    "Respondents": [
        len(frame),
        baseline_2015.sum(),
        (baseline_2015 & followup_2019).sum(),
        (baseline_2015 & followup_2019 & valid_target).sum(),
        (baseline_2015 & followup_2019 & valid_target & valid_time).sum(),
    ],
})

display(cohort_counts)

# Copy the dataset for modeling
eligible_dataset = frame.loc[
    baseline_2015 & followup_2019 & valid_target & valid_time,
    [id_col, *feature_cols, target_col]
].copy()

# Gates
assert eligible_dataset[id_col].is_unique # Check that each respondent is unique
assert eligible_dataset[target_col].isin([0, 1]).all() # Check that the target is binary


,Stage,Respondents
0,Original respondents,8984
1,Completed 2015 interview,7103
2,Completed both interviews,6402
3,Valid 2019 chronic-pain response,6389
4,Valid interview chronology,6389


### 3.2 Clean feature values

In [9]:
# Replace negative survey codes with missing values
for col in feature_cols:
    eligible_dataset[col] = eligible_dataset[col].mask(eligible_dataset[col] < 0)

# Check the valid response ranges
age = "CV_AGE_INT_DATE__2015"
health = "YHEA-100__2015"

activity_cols = [
    "YHEA-600AA__2015",
    "YHEA-600BA__2015",
    "YHEA-600CA__2015",
]

eligible_dataset[age] = eligible_dataset[age].where(
    eligible_dataset[age].between(30, 36)
)

eligible_dataset[health] = eligible_dataset[health].where(
    eligible_dataset[health].isin([1, 2, 3, 4, 5])
)

for col in activity_cols:
    eligible_dataset[col] = eligible_dataset[col].where(
        eligible_dataset[col].isin([0, 1, 2])
    )

# Examine missing predictor values
missing_summary = pd.DataFrame({
    "Missing": eligible_dataset[feature_cols].isna().sum(),
    "Percent missing (%)": (
        eligible_dataset[feature_cols].isna().mean() * 100
    ).round(2),
})

display(missing_summary)


,Missing,Percent missing (%)
CV_AGE_INT_DATE__2015,0,0.00
YHEA-100__2015,8,0.13
YHEA-600AA__2015,34,0.53
YHEA-600BA__2015,80,1.25
YHEA-600CA__2015,37,0.58
